# 02｜拆解现有选股策略
本练习调用实际因子、信号和组合函数，展示 2020 年短区间前五只股票的决策。每个 Notebook 使用独立内核，从本页第一格开始即可运行。

In [ ]:
from pathlib import Path
import sys, pandas as pd
ROOT = next((p.resolve() for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src" / "qlib_quant").is_dir()), None)
if ROOT is None: raise RuntimeError("请打开 D:/qlib_quant 项目目录。")
sys.path.insert(0, str(ROOT / "src"))
from qlib_quant.config import effective_config
from qlib_quant.runner import warmup_window
from qlib_quant.data.db import iter_stock_frames
from qlib_quant.factors.technical import add_technical_features
from qlib_quant.signals.legacy_v1 import score_signals
from qlib_quant.signals.legacy_v1 import signal_names, legacy_signal_names, signal_descriptions
from qlib_quant.portfolio.baseline import select_equal_weight
cfg = effective_config()
cfg["backtest"].update(start="2020-01-01", end="2020-02-28", max_stocks=5, max_rows=0, max_positions=3)
cfg["signals"]["enabled"] = list(legacy_signal_names)
print("原有启用信号:", len(cfg["signals"]["enabled"]), "预热观测:", warmup_window(cfg))

## 原有规则目录
每条规则输出真假，命中加 1 分。新增的教学信号不在原有 11 条之中，默认关闭。

In [ ]:
display(pd.DataFrame([{"信号": n, "中文说明": signal_descriptions[n],
                           "原策略默认启用": n in cfg["signals"]["enabled"]}
                          for n in signal_names]))

In [ ]:
warmup = warmup_window(cfg)
frames, stock_metadata = [], []
for frame, prior_count, index, total in iter_stock_frames(
        cfg["data"]["database"], cfg["backtest"]["start"], cfg["backtest"]["end"],
        warmup, cfg["backtest"]["max_stocks"]):
    code = str(frame.iloc[0]["code"])
    stock_metadata.append({"code": code, "warmup_rows": prior_count})
    featured = add_technical_features(frame, windows=cfg["factors"]["windows"])
    scored = score_signals(featured, cfg["signals"]["thresholds"], cfg["signals"]["enabled"])
    frames.append(scored.loc[scored.date.between(cfg["backtest"]["start"], cfg["backtest"]["end"])])
prepared = pd.concat(frames, ignore_index=True)
print("股票:", [x["code"] for x in stock_metadata],
      "绩效期行情行:", len(prepared), "预热:", warmup)
display(prepared.head())

## 从信号追到组合
优先检查最近一个确有候选的交易日；若短样本没有命中，则显示最后一个交易日。没有候选是可接受的真实结果，不要编造交易。

In [ ]:
any_by_day = prepared.groupby("date").signal.any()
active = any_by_day[any_by_day]
day_name = active.index[-1] if len(active) else prepared.date.max()
day = prepared.loc[prepared.date == day_name].copy()
print("交易日:", day_name, "候选数:", int(day.signal.sum()))
display(day[["date","code",*legacy_signal_names,"signal_score","signal"]]
        .sort_values(["signal_score","code"], ascending=[False, True]))
targets = select_equal_weight(day, max_positions=3)
print("下一交易日目标权重")
display(targets[["code","signal_score","target_weight"]])

## 练习与完成标准
1. 指出一个候选股票命中的规则和总分；若当天没有候选，记录该事实。
2. 找一个落选股票，区分未命中和未进入前三名。
3. 解释：因子是数值，信号是条件，排序决定先选谁，组合决定目标仓位。
4. 回网页打开“结果与实验”，从生效参数和每日交易查看更长的决策过程。

放量阈值填 2 表示今日量至少为昨日的 3 倍。YAML 中的 `price_rise_min/max` 尚未接入，编辑它们不会改变回测。